In [ ]:
# !pip install pyarrow
# !pip install fastparquet
# !pip install einops

In [ ]:
import sys
sys.path.append("..") 

import os
import pandas as pd
import numpy as np
from scripts.embedding_pipeline_st import EmbeddingPipeline
from scripts.embedding_pipeline_nomic import NomicEmbeddingPipeline
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
input_dir = "../data/"
output_dir = os.path.expanduser("~/data/outputs/")
os.makedirs(output_dir, exist_ok=True)

In [ ]:
df = pd.read_parquet(input_dir + 'amazon_products.parquet',  engine='fastparquet')
print("Data loaded successfully!")

In [ ]:
pipeline = EmbeddingPipeline(model_name='all-mpnet-base-v2')

# Run and Save
embeddings = pipeline.compute_embeddings(
    df=df, 
    column='title', 
    batch_size=128, 
    save_path=save_path
)

In [ ]:
def test_embeddings(df, embeddings, query_index, top_k=5):
    # Get the title and embedding for the "query" product
    query_title = df.iloc[query_index]['title']
    query_vec = embeddings[query_index].reshape(1, -1)
    
    print(f"--- QUERY PRODUCT (Index {query_index}) ---")
    print(f"Title: {query_title}\n")
    
    scores = cosine_similarity(query_vec, embeddings).flatten()
    indices = np.argsort(scores)[-(top_k + 1):][::-1]
    
    print(f"--- TOP {top_k} SIMILAR PRODUCTS ---")
    for i in indices:
        if i == query_index:
            continue
            
        similarity = scores[i]
        match_title = df.iloc[i]['title']
        print(f"[{similarity:.4f}] {match_title}")

In [ ]:
test_index = 100 
test_embeddings(df, embeddings, query_index=test_index)

#### Running Nomic embedding pipeline

In [ ]:
# Initialize the Nomic pipeline
nomic_pipeline = NomicEmbeddingPipeline(model_name='nomic-ai/nomic-embed-text-v1.5')

# Define save path
save_path = 'outputs/title_embeddings_nomic.npy'

# Compute and Save Embeddings
nomic_embeddings = nomic_pipeline.compute_embeddings(
    df=df,
    column='title',
    batch_size=64,
    save_path=save_path
)

print(f"Computation complete. Embedding shape: {nomic_embeddings.shape}")